In [19]:
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
import torch.nn as nn

In [20]:
df = pd.read_csv(r"fmnist_small.csv")
df.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,9,0,0,0,0,0,0,0,0,0,...,0,7,0,50,205,196,213,165,0,0
1,7,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,1,0,0,0,...,142,142,142,21,0,3,0,0,0,0
3,8,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,8,0,0,0,0,0,0,0,0,0,...,213,203,174,151,188,10,0,0,0,0


In [21]:
x = df.iloc[:, 1:]
y = df.iloc[:,0]

In [22]:
x

,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,pixel10,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,0,0,0,0,0,0,0,0,0,0,...,0,7,0,50,205,196,213,165,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,1,0,0,0,0,...,142,142,142,21,0,3,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,213,203,174,151,188,10,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5995,0,0,0,0,0,0,0,0,0,1,...,69,12,0,0,0,0,0,0,0,0
5996,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5997,0,0,0,0,0,0,0,0,0,0,...,39,47,2,0,0,29,0,0,0,0
5998,0,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0


In [23]:
y

0       9
1       7
2       0
3       8
4       8
       ..
5995    1
5996    5
5997    8
5998    4
5999    8
Name: label, Length: 6000, dtype: int64

In [24]:
x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.2,random_state=42)

In [25]:
x_train = x_train/255.0
x_test = x_test/255.0

In [26]:
class CustomDataset(Dataset):

    def __init__(self, features, labels):
        self.features = torch.tensor(features.values, dtype=torch.float32)
        self.labels = torch.tensor(labels.values, dtype=torch.long)

    def __len__(self):
        return len(self.features)

    def __getitem__(self, index):
        return self.features[index], self.labels[index]

In [27]:
train_dataset = CustomDataset(x_train, y_train)
test_dataset = CustomDataset(x_test, y_test)

In [28]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, pin_memory=True)

In [34]:
class MyNN(nn.Module):

    def __init__(self, num_features):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(num_features, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.5),

            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.5),

            nn.Linear(64, 10),
            nn.Softmax()
        )

    def forward(self, features):
        return self.network(features)    


In [35]:
learning_rate = 0.1
epoch = 100
loss_function = nn.CrossEntropyLoss()


In [36]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [37]:
model = MyNN(x_train.shape[1])

model = model.to(device)

optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate, weight_decay=0.01)

In [38]:
for epoch in range(epoch):
    total_loss = 0
    for features, lables in train_loader:
        features, lables = features.to(device), lables.to(device)
        y_pred = model(features)

        loss = loss_function(y_pred, lables)

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss/len(train_loader)    

    print(epoch+1, avg_loss)    

c:\Desktop\PyTorch\.venv\Lib\site-packages\torch\nn\modules\module.py:1739: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  return self._call_impl(*args, **kwargs)


1 2.11812784353892
2 1.941015711625417
3 1.8952817734082539
4 1.8731733632087708
5 1.8482851163546243
6 1.8291568716367086
7 1.8241901477177938
8 1.816572425365448
9 1.8190902153650919
10 1.8165188137690227
11 1.8120118061701456
12 1.8080764532089233
13 1.8066135541598003
14 1.8143057139714558
15 1.8095705485343934
16 1.8067014821370442
17 1.8154333877563475
18 1.8138673575719197
19 1.811422781944275
20 1.812092249393463
21 1.8068712695439657
22 1.81354340950648
23 1.8132598765691121
24 1.80937326669693
25 1.8097391017278035
26 1.8123832480112712
27 1.8098716243108113
28 1.808240181605021
29 1.8052660489082337
30 1.8066344038645425
31 1.811878788471222
32 1.80415296792984
33 1.8071493665377298
34 1.8115548133850097
35 1.8063868109385173
36 1.805686539808909
37 1.8034556571642557
38 1.8031307633717855
39 1.8049837748209636
40 1.8060348614056905
41 1.8042766722043355
42 1.804663151105245
43 1.8092313822110493
44 1.8096608026822407
45 1.806305300394694
46 1.8103854084014892
47 1.801471950

In [42]:
model.eval()

MyNN(
  (network): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=True)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.5, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.5, inplace=False)
    (8): Linear(in_features=64, out_features=10, bias=True)
    (9): Softmax(dim=None)
  )
)

In [43]:
total = 0
correct = 0


with torch.no_grad():
    for features, labels in train_loader:
        features, labels = features.to(device), labels.to(device)

        y_pred = model(features)
        _, predicted = torch.max(y_pred, 1)
        total+=lables.size(0)
        correct+=(predicted==labels).sum().item()
        
accuracy = correct/total
print(accuracy)        

0.7785416666666667


In [44]:
total = 0
correct = 0


with torch.no_grad():
    for features, labels in test_loader:
        features, labels = features.to(device), labels.to(device)

        y_pred = model(features)
        _, predicted = torch.max(y_pred, 1)
        total+=lables.size(0)
        correct+=(predicted==labels).sum().item()
        
accuracy = correct/total
print(accuracy)        

0.7425986842105263


ElaticNet

In [45]:
# l1_lambda = 0.001
# l2_lambda = 0.001

# for epoch in range(epochs):

#     for features, labels in train_loader:

#         y_pred = model(features)

#         original_loss = loss_function(
#             y_pred,
#             labels.view(-1,1)
#         )

#         # L1 penalty
#         l1_penalty = sum(
#             p.abs().sum()
#             for p in model.parameters()
#         )

#         # L2 penalty
#         l2_penalty = sum(
#             (p ** 2).sum()
#             for p in model.parameters()
#         )

#         # Total loss
#         loss = (
#             original_loss
#             + l1_lambda * l1_penalty
#             + l2_lambda * l2_penalty
#         )

#         optimizer.zero_grad()

#         loss.backward()

#         optimizer.step()